# Pond Location Analysis - Nellore & Eluru Regions
This notebook filters and plots pond locations using coordinates and village names.

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np
import time

# Load the data
file_path = 'data/Shared_ 2026 Github ARA Pond IDs Key.csv'
df = pd.read_csv(file_path)

# Display initial info
print(f"Total ponds: {len(df)}")
df.head()

Total ponds: 304


,internal_pond_id,public_pond_id,region,latitude,longitude,village,farmer_name
0,NL-BAN1,pond_43720001,Nellore,14.476104,80.109875,Indukurpeta,B. Bhanu Prakesh
1,NL-BAN2,pond_2f7733e2,Nellore,14.476550,80.109926,Indukurpeta,B. Bhanu Prakesh
2,NL-BAS1,pond_c0b647c8,Nellore,NaN,NaN,Somarajupalli,Basheer
3,NL-BJD1,pond_d1de1045,Nellore,14.471758,80.149839,Somarajupalli,Bathina Jagadeesh
4,NL-BNG1,pond_c30c0534,Nellore,14.521440,80.122916,Komarika,B. Nageswar Rao


## 2. Filter Valid Geographic Data
Filter out ponds which have no latitude/longitude AND no village name.

In [3]:
# Filter criteria: Keep if (Lat AND Lon are present) OR (Village is present)
# Note: In the data latitudes/longitudes are numeric (possibly NaN) and village is string (possibly empty)

# Handle empty latitude/longitude/village
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Filter logic
mask = (df['latitude'].notna() & df['longitude'].notna()) | (df['village'].notna() & df['village'].str.strip().ne(''))
df_filtered = df[mask].copy()

print(f"Ponds after filtering (must have coordinates or village): {len(df_filtered)}")
print(f"Dropped {len(df) - len(df_filtered)} ponds.")
df_filtered.head()

Ponds after filtering (must have coordinates or village): 252
Dropped 52 ponds.


,internal_pond_id,public_pond_id,region,latitude,longitude,village,farmer_name
0,NL-BAN1,pond_43720001,Nellore,14.476104,80.109875,Indukurpeta,B. Bhanu Prakesh
1,NL-BAN2,pond_2f7733e2,Nellore,14.476550,80.109926,Indukurpeta,B. Bhanu Prakesh
2,NL-BAS1,pond_c0b647c8,Nellore,NaN,NaN,Somarajupalli,Basheer
3,NL-BJD1,pond_d1de1045,Nellore,14.471758,80.149839,Somarajupalli,Bathina Jagadeesh
4,NL-BNG1,pond_c30c0534,Nellore,14.521440,80.122916,Komarika,B. Nageswar Rao


## 3. Geocode Villages with Missing Coordinates
For ponds that have a village name but are missing latitude/longitude, we assign a representative coordinate for that village within the region.
Instead of calling a live geocoding API which might fail or be slow, we'll use a local mapping of unique villages to their mean coordinates (if available) or placeholders for visualization.

In [4]:
# Step: Fill in missing coordinates using the average for the same village (if any has coords)
# Calculate village centroids
village_centroids = df_filtered.dropna(subset=['latitude', 'longitude']).groupby('village')[['latitude', 'longitude']].mean()

# Update missing coords
for idx, row in df_filtered[df_filtered['latitude'].isna()].iterrows():
    village = row['village']
    if village in village_centroids.index:
        df_filtered.loc[idx, 'latitude'] = village_centroids.loc[village, 'latitude']
        df_filtered.loc[idx, 'longitude'] = village_centroids.loc[village, 'longitude']

print(f"Ponds remaining with NaN coordinates: {df_filtered['latitude'].isna().sum()}")
df_filtered_ready = df_filtered.dropna(subset=['latitude', 'longitude'])
print(f"Ponds ready for plotting: {len(df_filtered_ready)}")

Ponds remaining with NaN coordinates: 0
Ponds ready for plotting: 252


In [9]:
# Summarize ponds that were assigned representative coordinates from village centroids
geocoded_summary = df.loc[
    df['latitude'].isna()
    & df['village'].notna()
    & df['village'].astype(str).str.strip().ne('')
    & df['village'].isin(village_centroids.index),
    ['public_pond_id', 'village']
].copy()

geocoded_summary['representative_latitude'] = geocoded_summary['village'].map(village_centroids['latitude'])
geocoded_summary['representative_longitude'] = geocoded_summary['village'].map(village_centroids['longitude'])
geocoded_summary = geocoded_summary.sort_values(['village', 'public_pond_id']).reset_index(drop=True)

print(
    f"Representative coordinates assigned to {len(geocoded_summary)} ponds across "
    f"{geocoded_summary['village'].nunique()} villages."
)
geocoded_summary

Representative coordinates assigned to 2 ponds across 2 villages.


,public_pond_id,village,representative_latitude,representative_longitude
0,pond_04e640c0,Komarika,14.516896,80.127804
1,pond_c0b647c8,Somarajupalli,14.481161,80.144407


## 4. Visualize Pond Locations on a Map
Create an interactive map using **folium** (Leaflet.js). Ponds are colored by region.
A native Leaflet scale bar (`control_scale=True`) is shown at the bottom-left and updates automatically as you zoom in or out.

In [14]:
import folium
import branca.colormap as cm

p_plottable = df_filtered_ready.copy()

center_lat = p_plottable['latitude'].mean()
center_lon = p_plottable['longitude'].mean()

# Create the map with a native Leaflet scale bar that updates on every zoom
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8,
    tiles='OpenStreetMap',
    control_scale=True   # <-- native dynamic scale bar
)

# Assign a distinct color to each region
regions = sorted(p_plottable['region'].dropna().unique())
palette = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
region_color = {r: palette[i % len(palette)] for i, r in enumerate(regions)}

# Add a circle marker for every pond
for _, row in p_plottable.iterrows():
    color = region_color.get(row.get('region'), '#7f7f7f')
    village_name = row.get('village', 'Unknown')
    popup_html = (
        f"<b>{row['public_pond_id']}</b><br>"
        f"Region: {row.get('region', '')}<br>"
        f"Village: {village_name}<br>"
        f"Farmer: {row.get('farmer_name', '')}"
    )
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        weight=1,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=row['public_pond_id'],
    ).add_to(m)

    # Add village name label
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:10px;color:{color};font-weight:bold;'
                f'white-space:nowrap;margin-left:10px;margin-top:-6px;">'
                f'{village_name}</div>'
            ),
            icon_size=(150, 20),
            icon_anchor=(0, 10),
        ),
    ).add_to(m)

# Legend (HTML overlay)
legend_html = """
<div style="position:fixed;bottom:40px;right:10px;z-index:9999;
            background:white;padding:10px 14px;border:2px solid #555;
            border-radius:6px;font-size:13px;line-height:1.6">
  <b>Region</b><br>
"""
for region, color in region_color.items():
    legend_html += f'  <span style="color:{color};font-size:16px">&#9679;</span> {region}<br>\n'
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

display(m)

## 5. Infer Missing Village Names (Nearest-Neighbour)
For ponds that have coordinates but **no village name**, find the nearest pond (crow-fly distance) that *does* have a village name and assign that village. The CSV is then overwritten with the inferred values.

In [10]:
import numpy as np

def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    """Crow-fly distance (km) from one point to an array of points."""
    R = 6371.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2_arr)
    dphi = phi2 - phi1
    dlam = np.radians(lon2_arr) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Reload the CSV fresh so in-memory mutations don't interfere
df_csv = pd.read_csv(file_path)
df_csv['latitude'] = pd.to_numeric(df_csv['latitude'], errors='coerce')
df_csv['longitude'] = pd.to_numeric(df_csv['longitude'], errors='coerce')

# ── Ponds with coordinates but no village ──────────────────────────────────
no_vil_mask = (
    df_csv['latitude'].notna() & df_csv['longitude'].notna() &
    (df_csv['village'].isna() | df_csv['village'].astype(str).str.strip().eq(''))
)
df_no_vil = df_csv[no_vil_mask].copy()
print(f"Ponds with coordinates but no village: {len(df_no_vil)}")

# ── Reference ponds: have coordinates AND a village name ──────────────────
ref_mask = (
    df_csv['latitude'].notna() & df_csv['longitude'].notna() &
    df_csv['village'].notna() & df_csv['village'].astype(str).str.strip().ne('')
)
df_ref = df_csv[ref_mask][['public_pond_id', 'village', 'latitude', 'longitude']].reset_index(drop=True)
ref_lats = df_ref['latitude'].values
ref_lons = df_ref['longitude'].values

# ── Nearest-neighbour inference ────────────────────────────────────────────
rows = []
for _, r in df_no_vil.iterrows():
    dists = haversine_km(r['latitude'], r['longitude'], ref_lats, ref_lons)
    ni = int(np.argmin(dists))
    rows.append({
        'internal_pond_id':  r['internal_pond_id'],
        'public_pond_id':    r['public_pond_id'],
        'region':            r['region'],
        'latitude':          r['latitude'],
        'longitude':         r['longitude'],
        'inferred_village':  df_ref.loc[ni, 'village'],
        'nearest_ref_pond':  df_ref.loc[ni, 'public_pond_id'],
        'distance_km':       round(float(dists[ni]), 3),
    })

df_inferred = pd.DataFrame(rows)

# ── Overwrite blank village cells in the CSV ───────────────────────────────
df_updated = df_csv.copy()
for _, r in df_inferred.iterrows():
    df_updated.loc[df_updated['internal_pond_id'] == r['internal_pond_id'], 'village'] = r['inferred_village']

df_updated.to_csv(file_path, index=False)
print(f"CSV saved → {file_path}  ({len(df_inferred)} village cells updated)\n")

print(f"Inferred villages for {len(df_inferred)} ponds:")
df_inferred[['public_pond_id', 'region', 'inferred_village', 'nearest_ref_pond', 'distance_km']]

Ponds with coordinates but no village: 42
CSV saved → data/Shared_ 2026 Github ARA Pond IDs Key.csv  (42 village cells updated)

Inferred villages for 42 ponds:


,public_pond_id,region,inferred_village,nearest_ref_pond,distance_km
0,pond_b11d4126,Eluru,Gogunta,pond_b36b9eee,0.820
1,pond_518460bf,Eluru,Ponangi,pond_6526e587,0.292
2,pond_7c2f9913,Eluru,Ponangi,pond_6526e587,0.365
3,pond_811d5777,Eluru,Gudivakalanka,pond_bed02137,0.341
4,pond_c1aade15,Eluru,Veeramma Kunta,pond_44f24b9a,1.263
5,pond_5aaa5b01,Eluru,Durgapuram,pond_dc98e449,0.640
6,pond_d452b0a2,Eluru,Sreeparru,pond_eb2903bd,0.215
7,pond_cc764bfd,Eluru,Jalipudi,pond_7917cb8e,0.114
8,pond_4389488f,Eluru,Sreeparru,pond_a80e42b6,0.591
9,pond_09b2c644,Eluru,Maanuru,pond_7fd480e0,0.175


In [12]:
import folium

center_lat2 = df_inferred['latitude'].mean()
center_lon2 = df_inferred['longitude'].mean()

m2 = folium.Map(
    location=[center_lat2, center_lon2],
    zoom_start=10,
    tiles='OpenStreetMap',
    control_scale=True,
)

villages2 = sorted(df_inferred['inferred_village'].unique())
palette2 = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
            '#8c564b', '#e377c2', '#17becf', '#bcbd22', '#7f7f7f']
village_color2 = {v: palette2[i % len(palette2)] for i, v in enumerate(villages2)}

for _, row in df_inferred.iterrows():
    color = village_color2[row['inferred_village']]
    popup_html = (
        f"<b>{row['public_pond_id']}</b><br>"
        f"Region: {row['region']}<br>"
        f"Inferred village: <b>{row['inferred_village']}</b><br>"
        f"Nearest ref pond: {row['nearest_ref_pond']} ({row['distance_km']} km)"
    )
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=1.5,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=row['public_pond_id'],
    ).add_to(m2)

    # Persistent village-name label offset to the right of the marker
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:10px;color:{color};font-weight:bold;'
                f'white-space:nowrap;margin-left:10px;margin-top:-6px;">'
                f'{row["inferred_village"]}</div>'
            ),
            icon_size=(160, 20),
            icon_anchor=(0, 10),
        ),
    ).add_to(m2)

# Legend
legend2 = (
    '<div style="position:fixed;bottom:40px;right:10px;z-index:9999;'
    'background:white;padding:10px 14px;border:2px solid #555;'
    'border-radius:6px;font-size:13px;line-height:1.6">'
    '<b>Inferred Village</b><br>'
)
for v, c in village_color2.items():
    legend2 += f'<span style="color:{c};font-size:16px">&#9679;</span> {v}<br>'
legend2 += '</div>'
m2.get_root().html.add_child(folium.Element(legend2))

display(m2)